### Import all the libraries

In [1]:
#Import all the libraries

import os
from dotenv import load_dotenv
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index
from rag_helper import RAGBase
from openai import OpenAI
from toyaikit.tools import Tools
from toyaikit.llm import OpenAIChatCompletionsClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIChatCompletionsRunner, DisplayingRunnerCallback

#load the .env file 

load_dotenv()

True

### Preparation

In [2]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)

### Q1. How many lesson pages

In [4]:
print(len(documents))

72


### Q2. Indexing and searching

In [5]:
index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(documents)

In [6]:
mistral_client = OpenAI(
    api_key=os.getenv("MISTRAL_API_KEY"),
    base_url="https://api.mistral.ai/v1"
)

In [7]:
question = "How does the agentic loop keep calling the model until it stops?"

In [8]:
search_results = index.search(question)
search_results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

### Q3. RAG

In [9]:
assistant = RAGBase(
    index=index,
    llm_client=mistral_client
)

In [10]:
response = assistant.rag(question)
response

("L'agent appelle le modèle en boucle jusqu'à ce qu'il n'y ait plus d'appels de fonction (tool calls) dans sa réponse.\n\nVoici comment cela fonctionne dans le contexte fourni :\n\n1. Le modèle reçoit l'historique complet des messages (y compris les résultats des outils précédents).\n2. Il décide s'il a besoin d'un nouvel appel de fonction (par exemple, une recherche) ou s'il peut répondre directement.\n3. Si le modèle retourne un appel de fonction (`function_call`), le code exécute l'outil correspondant, ajoute le résultat à l'historique des messages, et relance le modèle avec ce nouvel historique.\n4. Ce processus se répète dans une boucle `while True` jusqu'à ce que le modèle retourne une réponse finale **sans** aucun appel de fonction.\n5. La boucle s'arrête lorsque le flag `has_function_calls` reste à `False` après traitement de la réponse du modèle.\n\nL'extrait clé est :\n```python\nwhile True:\n    # ... (appel du modèle)\n    if has_function_calls == False:\n        break\n```

### Q4. Chunking

In [11]:
chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))

295


### Q5. RAG with chunking

In [12]:
index.fit(chunks)

In [13]:
assistant = RAGBase(
    index=index,
    llm_client=mistral_client,
)

In [14]:
response = assistant.rag('How does the agentic loop keep calling the model until it stops?')
response

("L'agent utilise une boucle `while True` qui continue d'appeler le modèle tant qu'il y a des appels de fonction (c'est-à-dire des outils à exécuter). Voici comment cela fonctionne en détail :\n\n1. **Condition de continuation** : La boucle s'exécute tant qu'il y a au moins un appel de fonction détecté (`has_function_calls == True`). Chaque itération vérifie si le modèle a demandé un appel d'outil.\n\n2. **Exécution du modèle** : À chaque itération, le modèle est appelé avec l'historique des messages mis à jour (incluant les résultats des outils précédents). Le modèle peut alors :\n   - **Retourner un appel de fonction** : Si le modèle demande un outil (par exemple, une recherche), l'agent exécute cet outil, ajoute le résultat à l'historique des messages, et relance une itération.\n   - **Retourner une réponse finale** : Si le modèle ne demande plus d'outil, la boucle s'arrête.\n\n3. **Condition d'arrêt** : La boucle s'interrompt uniquement lorsque le modèle retourne une réponse finale

### Q6. Turning it into an agent

In [15]:
def search(query: str) -> dict[str, str]:
    """
    Search the chunks for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
    )

In [16]:
instructions = """
    You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
""".strip()

In [ ]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [ ]:
llm_client = OpenAIChatCompletionsClient(
    model="mistral-small-latest",
    client=mistral_client
)

In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIChatCompletionsRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [ ]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


Feature,Plain RAG,Agentic Loop
Interaction Type,Single-turn,Multi-turn
Adaptability,Limited (fixed pipeline),High (adapts based on results)
Memory,None (no history retention),Yes (conversation history preserved)
Tool Usage,None,"Can call tools (e.g., search, web)"
Decision-Making,Predefined steps,Dynamic (LLM decides next steps)
Use Case,Simple Q&A with known answers,Complex queries requiring reasoning
Example,"""What is RAG?""","""Explain RAG and compare it to other methods"""
